In [2]:
from pyspark.sql import functions as F
from data_generator import get_spark, generate_skewed_table
import time

spark = get_spark("Case04_Salting")

skewed_df = generate_skewed_table(spark, num_rows=2_000_000, hot_keys=5, hot_key_ratio=0.6)
skewed_df.cache().count()

2000000

In [3]:
print("Data Distribution (top 10 keys by count):")
skewed_df.groupby("account_number").count().orderBy(F.desc("count")).show()

Data Distribution (top 10 keys by count):


+--------------+------+
|account_number| count|
+--------------+------+
|             0|240000|
|             1|240000|
|             3|240000|
|             4|240000|
|             2|240000|
|            44|     8|
|             9|     8|
|           296|     8|
|            24|     8|
|           187|     8|
|           313|     8|
|           467|     8|
|           336|     8|
|           375|     8|
|           444|     8|
|           675|     8|
|           465|     8|
|           441|     8|
|           562|     8|
|           691|     8|
+--------------+------+
only showing top 20 rows



# print(" Data Distribution (top 10 keys by count):")

In [5]:
start=time.time()
result_bad=(
    skewed_df
    .groupBy("account_number")
    .agg(
        F.sum("value").alias("total_value"),
        F.count("*").alias("event_count")
    )
)
result_bad.write.mode("overwrite").parquet("/tmp/case04_bad")
print(f" Direct aggregation: {time.time() - start:.1f}s")

26/04/12 11:54:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/12 11:54:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/12 11:54:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/12 11:54:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/12 11:54:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers


 Direct aggregation: 2.9s


26/04/12 11:54:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/12 11:54:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/12 11:54:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/12 11:54:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


# GOOD: Salt the key → aggregate → unsalt

In [9]:
NUM_SALTS = 10
start=time.time()
salted_df=skewed_df.withColumn("salt",(F.rand()*NUM_SALTS).cast("int"))
partial_agg=(
    salted_df
    .groupBy("account_number","salt")
    .agg(
        F.sum("value").alias("partial_sum"),
        F.count("*").alias("partial_count")
    )
)


result_good = (
    partial_agg
    .groupBy("account_number")
    .agg(
        F.sum("partial_sum").alias("total_value"),
        F.sum("partial_count").alias("event_count")
    )
)
result_good.write.mode("overwrite").parquet("/tmp/case04_good")
print(f"Salted aggregation: {time.time() - start:.1f}s")

Salted aggregation: 1.9s
